In [2]:
import pandas as pd
# Load recipe data from Excel
file_path = "merged_recipes.xlsx"  # Update the file name if needed
recipe_df = pd.read_excel(file_path)

# Display the first few rows to confirm
print(recipe_df.head())

           Recipe Name                                        Ingredients  \
0                Pulao  Rice, Mutton/Beef, Whole Spices, Onions, Ginge...   
1               Tahari  Rice, Potatoes, Onions, Tomatoes, Green Chilie...   
2          Saag Chawal  Rice, Spinach, Mustard Greens, Green Chilies, ...   
3         Machli Pulao  Rice, Fish, Yogurt, Tomatoes, Onions, Green Ch...   
4  Chicken Qorma Pulao  Rice, Chicken, Yogurt, Qorma Masala, Onions, G...   

  Cuisine Region      Cooking Method  \
0         Punjab  Boiling, Simmering   
1         Punjab     One-Pot Cooking   
2         Punjab   Boiling, Sautéing   
3          Sindh  Boiling, Simmering   
4         Punjab  Boiling, Simmering   

                                              Recipe Category  
0  Sauté onions and whole spices in ghee. Add mea...     Rice  
1  Sauté onions and spices in ghee. Add potatoes ...     Rice  
2  Boil rice separately. Sauté spinach and mustar...     Rice  
3  Cook fish with spices and tomatoes. B

In [3]:
def get_matching_recipes(available_ingredients, recipe_df):
    available_ingredients_set = set(available_ingredients.lower().split(','))
    
    matching_recipes = []
    for idx, row in recipe_df.iterrows():
        recipe_ingredients = set(row['Ingredients'].lower().split(','))
        common_ingredients = available_ingredients_set.intersection(recipe_ingredients)
        
        if common_ingredients:
            matching_recipes.append(row)
    
    return matching_recipes

# Example of getting recipes based on available ingredients
available_ingredients = "chicken, rice, onions"
matching_recipes = get_matching_recipes(available_ingredients, recipe_df)

for recipe in matching_recipes:
    print(recipe['Recipe Name'])


Pulao
Tahari
Machli Pulao
Chicken Qorma Pulao
Chicken Shashlik Biryani
Aloo Keema Pulao
Kachhi Biryani
Tandoori Chicken Biryani
Murg Methi Biryani
Anda Chawal
Paya Biryani
Achari Chicken Biryani
Mutton Karahi Pulao
Hara Masala Biryani
Bhuna Gosht Biryani
Karachi Chicken Biryani
Chana Biryani
Korma Biryani
Nihari Biryani
Saffron Prawn Biryani
Bhindi Gosht Pulao
Baingan Biryani
Methi Chicken Biryani
Kaleji Biryani
Chicken Saag Pulao
Tikka Biryani
Masala Chicken Pulao
Karachi Biryani
Peshawari Pulao
Chicken Biryani with Mint
Green Pulao
Bhuni Biryani
Punjabi Vegetable Biryani
Biryani with Raisins
Saffron Chicken Pulao
Hyderabadi Chicken Pulao
Biryani with Mint
Karachi Tikka Biryani
Chicken Biryani with Peas
Tomato Chicken Pulao
Jeera Pulao
Chicken Keema Pulao
Dum Ka Chicken Biryani
Chicken Pulao with Lemon
Chingri Pulao
Kebab Pulao
Palak Keema Pulao
Chicken Tikka Pulao
Cheesy Chicken Pulao
Pea Pulao
Vegetable Masala Pulao
Corn Pulao
Malai Chicken Pulao
Chicken Biryani with Raisins
Achari 

In [4]:
ingredient_substitutes = {
    'chicken': 'tofu',
    'beef': 'lentils',
    'milk': 'coconut milk',
    'yogurt': 'plant-based yogurt'
}

def substitute_ingredients(ingredients, ingredient_substitutes):
    ingredients_list = ingredients.split(',')
    substituted_ingredients = [
        ingredient_substitutes.get(ingredient.strip(), ingredient.strip()) for ingredient in ingredients_list
    ]
    return ', '.join(substituted_ingredients)

# Example of substituting chicken with tofu
recipe_with_substitutes = substitute_ingredients('chicken, rice, onions', ingredient_substitutes)
print(recipe_with_substitutes)  # Output: tofu, rice, onions


tofu, rice, onions


In [5]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# Load pre-trained GPT model and tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2')

def generate_recipe(prompt):
    inputs = tokenizer.encode(prompt, return_tensors='pt')
    outputs = model.generate(inputs, max_length=200, num_return_sequences=1, no_repeat_ngram_size=2, top_k=50, top_p=0.95)
    recipe = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return recipe

# Example of generating a recipe based on available ingredients
prompt = "Generate a Pakistani recipe with chicken, rice, onions, and garlic."
generated_recipe = generate_recipe(prompt)
print(generated_recipe)


C:\Users\user\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:157: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see act

Generate a Pakistani recipe with chicken, rice, onions, and garlic.

Cook the chicken in a large pot over medium heat until it is tender. Add the onions and cook for about 5 minutes, or until the garlic is softened. Remove from heat and let cool.


In [7]:
# Full Recipe Generator System

def dynamic_recipe_generator(available_ingredients, recipe_df, ingredient_substitutes):
    # Find matching recipes based on available ingredients
    matching_recipes = get_matching_recipes(available_ingredients, recipe_df)
    
    if not matching_recipes:
        # If no exact matches, try generating a recipe dynamically
        prompt = f"Generate a Pakistani recipe with {available_ingredients}."
        generated_recipe = generate_recipe(prompt)
        generated_recipe = adjust_quantities(generated_recipe, servings=4)  # Adjust for servings
        return generated_recipe
    
    # Otherwise, return the first matching recipe
    best_match = matching_recipes[0]
    adjusted_recipe = substitute_ingredients(best_match['Ingredients'], ingredient_substitutes)
    return f"Recipe: {best_match['Recipe Name']}\nIngredients: {adjusted_recipe}\nInstructions: {best_match['Recipe']}"

# Example usage:
available_ingredients = "chicken, rice, onions"
recipe = dynamic_recipe_generator(available_ingredients, recipe_df, ingredient_substitutes)
print(recipe)


Recipe: Pulao
Ingredients: Rice, Mutton/Beef, Whole Spices, Onions, Ginger, Garlic, Green Chilies, Stock, Salt, Ghee
Instructions: Sauté onions and whole spices in ghee. Add meat and cook with stock until tender. Add rice, simmer until cooked. Serve with raita and salad.


In [9]:
import pandas as pd

# Load the dataset
df = pd.read_excel("merged_recipes.xlsx")

# Create a text-based dataset (prompt + recipe instructions)
prompts_responses = []
for _, row in df.iterrows():
    prompt = f"Generate a recipe with ingredients: {row['Ingredients']}."
    response = f"Instructions: {row['Recipe']}"
    prompts_responses.append(f"{prompt}\n{response}")

# Save the prompts_responses as a text file
with open("recipe_data.txt", "w") as f:
    for item in prompts_responses:
        f.write("%s\n\n" % item)

print("Text dataset for training saved.")


Text dataset for training saved.
